## Проект: Retrieval-Augmented Generation (RAG) на данных StackOverflow
## Описание проекта

Цель проекта — построить систему поиска и генерации ответов на основе данных StackOverflow с использованием подхода Retrieval-Augmented Generation (RAG).

В рамках проекта реализован полный пайплайн подготовки данных, разбиения документов на чанки, генерации эмбеддингов и хранения их в векторной базе данных PostgreSQL с расширением pgvector.

Подход RAG позволяет находить релевантные фрагменты текста с помощью векторного поиска и использовать их в качестве контекста для генерации ответа языковой моделью.

In [1]:
from src.core.config import Settings
from src.pipeline_loader import LoaderPipeline, VectorStore
from src.pipeline_retrieval import RetrievalPipeline
import psycopg2
from src.indexing.embedder import EmbeddingGenerator
from src.reranker.reranker import Reranker
from src.indexing.vector_store import VectorStore
from src.core.logging import setup_logging
from langgraph.graph import StateGraph, START, END
from src.core.queries import INSERT_CHUNK_QUERY, INSERT_CHUNK_QUERY_TEST

import logging
settings = Settings()
embedding_generator = EmbeddingGenerator()
encoder = Reranker(settings)
db_saver = VectorStore(settings)

setup_logging(settings)

logger = logging.getLogger(__name__)
logger.info("Логгер запущен")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-27 12:43:44,142 | INFO | __main__ | Логгер запущен


In [2]:
pipeline_loader = LoaderPipeline(settings, embedding_generator)
pipeline_retrieval = RetrievalPipeline(settings, embedding_generator, encoder)

2026-03-27 12:43:44,165 | INFO | src.reranker.reranker | Загрузка reranker-модели: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-27 12:43:46,717 | INFO | src.generation.llm_client | Загрузка LLM-модели: google/flan-t5-large


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-27 12:43:53,760 | INFO | src.generation.llm_client | LLM загружена на устройство: cuda


TypeError: StateGraph.__init__() missing 1 required positional argument: 'state_schema'

In [8]:
data = pipeline_loader.run()

with psycopg2.connect(**settings.DB_PARAMS) as conn:
    with conn.cursor() as cursor:
        rows = db_saver.build_insert_rows(data)
        db_saver.insert_rows(rows, conn, cursor, INSERT_CHUNK_QUERY_TEST)

2026-03-27 12:27:34,170 | INFO | src.core.langgragh | Node: DataLoader
2026-03-27 12:27:34,172 | INFO | src.ingestion.data_loader | Начало загрузки входных данных
2026-03-27 12:27:34,173 | INFO | src.ingestion.data_loader | Загрузка данных из data/raw/Questions.csv.
2026-03-27 12:27:34,174 | INFO | src.ingestion.csv_loader | Чтение заголовка файла: data/raw/Questions.csv
2026-03-27 12:27:34,181 | INFO | src.ingestion.csv_loader | Загрузка файла data/raw/Questions.csv. Колонок для чтения: 4. Ограничение строк: 200.
2026-03-27 12:27:34,188 | INFO | src.ingestion.csv_loader | Данные из файла data/raw/Questions.csv успешно загружены, 200 строк.
2026-03-27 12:27:34,189 | INFO | src.ingestion.data_loader | Получено 200 строк.
2026-03-27 12:27:34,189 | INFO | src.ingestion.data_loader | Загрузка данных из data/raw/Answers.csv.
2026-03-27 12:27:34,191 | INFO | src.ingestion.csv_loader | Чтение заголовка файла: data/raw/Answers.csv
2026-03-27 12:27:34,196 | INFO | src.ingestion.csv_loader | Заг

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

2026-03-27 12:27:34,744 | INFO | src.indexing.vector_store | Получено строк: 76
2026-03-27 12:27:34,745 | INFO | src.indexing.vector_store | Все необходимые колонки присутствуют
2026-03-27 12:27:34,749 | INFO | src.indexing.vector_store | Сформировано 76 строк для вставки
2026-03-27 12:27:34,750 | INFO | src.indexing.vector_store | Пример строки: ('q80_a124_c0', 80, 124, 0, 'SQLStatement.execute() - multiple queries in one statement', 'actionscript-3, air, flex', 26, 12, "Title : SQLStatement.execute ( ) - multiple queries in one statement Tags : actionscript-3 , air , flex Question : I 've written a database generation script in SQL and want to execute it in my Adobe AIR application : Create Table tRole ( roleID integer Primary Key , roleName varchar ( 40 ) ) ; Create Table tFile ( fileID integer Primary Key , fileName varchar ( 50 ) , fileDescription varchar ( 500 ) , thumbnailID integer , fileFormatID integer , categoryID integer , isFavorite boolean , dateAdded date , globalAccessC

### 6. Поиск

In [ ]:
query = "How to print Hello World in Python?"
answer = pipeline_retrieval.run(query)

2026-03-27 09:51:29,994 | INFO | src.pipeline_retrieval | Шаг 1: Формирование запроса в базу данных.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-27 09:51:30,196 | INFO | src.pipeline_retrieval | Эмбеддинг запроса [0.019435974, 0.08953558, -0.04974294, 0.015652642, -0.026656052, -0.15823485, -0.018088067, -0.01309944, -0.10564635, -0.017737221, 0.08653759, -0.038171068, 0.036263745, -0.002315856, 0.030325005, -0.05694112, -0.038167913, 0.0004744887, 0.03448758, -0.09435223, 0.054428585, 0.07680798, 0.066108495, 0.033949286, -0.005945955, -0.038748406, 0.021388156, 0.09014582, 0.010623463, 0.0031868836, 0.02249712, 0.02411088, 0.13482465, 0.00031163244, 0.08159091, 0.06244769, 0.018858595, -0.066973194, -0.020442745, 0.010359467, 0.047454935, -0.009492643, -0.0372886, -0.011283675, -0.06860958, 0.002335099, -0.13637754, 0.063536495, 0.07791794, 0.03873432, -0.036505617, -0.036837187, -0.040071495, -0.058308356, 0.08262671, 0.026748298, 0.040195115, 0.00022049417, -0.002612344, -0.099448204, -0.08233489, 0.025788054, -0.034292474, -0.006652218, 0.08424411, -0.033406913, -0.03525175, 0.08554044, -0.034833357, -0.042910498, 

Чанки [{'chunk_id': 'q10395430_a10395681_c4', 'chunk_text': "print `` Hello World ''", 'distance': 0.12231981754302979, 'question_id': 10395430, 'answer_id': 10395681, 'chunk_index': 4, 'title': 'Are Vala and Genie production ready?', 'tags': 'c, genie, gobject, vala'}, {'chunk_id': 'q6953550_a6953626_c0', 'chunk_text': "Title : Why do ` print ( `` Hello , World ! `` ) ` and ` print ( `` Hello '' , `` World ! `` ) ` produce different outputs ? Tags : python Question : I just installed Python 2.7.2 on Windows XP with the idea of learning how to program . Several of the tutorial books I 'm using give examples of print commands which , when I try them , I get different answers . I expected both of these to return the same thing - > > > print ( `` Hello , World ! '' ) Hello , World ! > > > print ( `` Hello '' , `` World '' ) ( 'Hello ' , 'World ' ) > > > I 've tried searching around for answers , but I 'm not even sure how to explain where I 'm", 'distance': 0.26851005962550867, 'question_

2026-03-27 09:51:46,683 | INFO | src.reranker.reranker | После rerank оставляем top-1
2026-03-27 09:51:46,685 | INFO | src.pipeline_retrieval | Всего чанков: 1
2026-03-27 09:51:46,686 | INFO | src.pipeline_retrieval | Шаг 4: Сборка контекста
2026-03-27 09:51:46,686 | INFO | src.generation.prompt_builder | Начало build_context. Получено чанков: 1
2026-03-27 09:51:46,687 | INFO | src.generation.prompt_builder | Собран context длиной 490 символов
2026-03-27 09:51:46,687 | INFO | src.pipeline_retrieval | Шаг 5: Сборка prompt
2026-03-27 09:51:46,702 | INFO | src.pipeline_retrieval | Шаг 6: Генерация ответа


Реранкер [{'chunk_id': 'q23609580_a23609686_c1', 'chunk_text': "this ? : Hello and welcome to my world I 'd like you to meet my family Thanks Best Answer : If you use Python 2 , print ( `` Hello and welcome to my world '' ) print print ( `` I 'd like you to meet my family '' ) Or if you use Python 3 , print ( `` Hello and welcome to my world '' ) print ( ) print ( `` I 'd like you to meet my family '' ) Or use \\n . This works with any version of Python . print ( `` Hello and welcome to my world\\n '' ) print ( `` I 'd like you to meet my family '' )", 'distance': 0.33522921800613403, 'question_id': 23609580, 'answer_id': 23609686, 'chunk_index': 1, 'title': 'Python spacing between outputted text in the shell', 'tags': 'python, shell', 'rerank_score': 8.334845542907715}]
Контекст this ? : Hello and welcome to my world I 'd like you to meet my family Thanks Best Answer : If you use Python 2 , print ( `` Hello and welcome to my world '' ) print print ( `` I 'd like you to meet my family 

2026-03-27 09:51:48,020 | INFO | src.generation.llm_client | Сгенерирован ответ длиной 68 символов
2026-03-27 09:51:48,021 | INFO | src.pipeline_retrieval | Pipeline завершён
2026-03-27 09:51:48,022 | INFO | src.core.config | Время выполнения функции 'run': 18.0284 секунд


Ответ Hello and welcome to my world I 'd like you to meet my family Thanks


In [5]:
print(answer)

Hello and welcome to my world '' ) print print (  I 'd like you to meet my family


In [ ]:
class LoaderPipeline:
    def __init__(self, settings, embedder):
        self.settings = settings
        self.data_loader = DataLoader(settings)
        self.preprocessing = Preprocessing()
        self.embedder = embedder
        self.chunker = Chunker()
        self.db_saver = VectorStore(settings)

    @time_decorator  # Применяем декоратор для замера времени
    def run(self):
        #----------------------------------------------------------------------------------------
        logger.info("Шаг 1: Загрузка данных")
        raw_data = self.data_loader.load_data()

        #----------------------------------------------------------------------------------------
        logger.info("Шаг 2: Предобработка")
        data = self.preprocessing.preprocess_data(raw_data.questions, raw_data.answers, raw_data.tags)
        logger.info(f"После preprocess: {len(data)} строк")

        #----------------------------------------------------------------------------------------
        logger.info("Шаг 3: Чанкинг")
        data["chunks"] = data["document_text"].apply(self.chunker.chunk_document)
        data["chunk_count"] = data["chunks"].apply(len)

        data = self.chunker.build_chunks_df(data)
        logger.info(f"Всего чанков: {len(data)}")

        #----------------------------------------------------------------------------------------
        logger.info("Шаг 4: Генерация эмбеддингов")
        embeddings = self.embedder.encode(data['chunk_text'].tolist())
        data['embedding'] = list(embeddings)
        
        logger.info("Pipeline завершён")
        return data

In [ ]:
def build_chunk(self, data):
    chunks_data = []

    for _, row in data.iterrows():
        # Разбиваем document_text на чанки
        chunks = self.chunker.chunk_document(row[Columns.DOCUMENT_TEXT.value])
        
        # Для каждого чанка создаем запись
        for idx, chunk in enumerate(chunks):
            chunk_data = {
                Columns.CHUNK_ID.value : f"q{row[Columns.QUESTION_ID.value]}_a{row[Columns.ANSWER_ID.value]}_c{idx}",
                Columns.QUESTION_ID.value : row[Columns.QUESTION_ID.value],
                Columns.ANSWER_ID.value : row[Columns.ANSWER_ID.value],
                Columns.CHUNK_INDEX.value : idx,
                Columns.TITLE.value : row[Columns.TITLE.value],
                Columns.TAGS.value : row[Columns.TAGS.value],
                Columns.QUESTION_SCORE.value : row[Columns.QUESTION_SCORE.value],
                Columns.ANSWER_SCORE.value: row[Columns.ANSWER_SCORE.value],
                Columns.CHUNK_TEXT.value: chunk
            }
            chunks_data.append(chunk_data)

    # Преобразуем список в DataFrame
    chunks_df = pd.DataFrame(chunks_data)

    # Вставляем данные в БД
    self.loader.insert_chunks_to_db(chunks_df)